# 🤖 AquaVidarbha — Complete Model Training Pipeline
## XGBoost + LSTM + Ensemble for Groundwater Depth Prediction

**Pipeline Steps:**
1. Data Loading & Preprocessing
2. Feature Engineering & Selection
3. Train/Test Split (Time-based)
4. **XGBoost** — Tabular ML Model
5. **Random Forest** — Baseline ML
6. **LSTM** — Deep Learning Sequence Model
7. **Ensemble** — Weighted Combination
8. Model Evaluation & Comparison
9. SHAP Feature Importance
10. Model Export for Production

In [ ]:
# ============================================================
# 1. IMPORTS
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import json
import joblib
from datetime import datetime

# Scikit-learn
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (mean_squared_error, mean_absolute_error, r2_score,
                             mean_absolute_percentage_error)
from sklearn.ensemble import RandomForestRegressor

# XGBoost
import xgboost as xgb

# Deep Learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (LSTM, GRU, Dense, Dropout, Input,
                                     Conv1D, MaxPooling1D, Flatten, BatchNormalization)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

COLORS = {
    'deep': '#355872', 'mid': '#7AAACE', 'light': '#9CD5FF',
    'crisis': '#C0392B', 'warn': '#E67E22', 'safe': '#27AE60'
}

# Create model output directory
os.makedirs('../models', exist_ok=True)
os.makedirs('../data/plots', exist_ok=True)

print(f'✅ TensorFlow version: {tf.__version__}')
print(f'✅ XGBoost version: {xgb.__version__}')
print(f'✅ GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

In [ ]:
# ============================================================
# 2. LOAD DATA
# ============================================================
df_ext = pd.read_csv('../data/vidarbha_groundwater_extended_v2.csv')
df_ext['date'] = pd.to_datetime(df_ext['date'])

df_model = pd.read_csv('../data/vidarbha_groundwater_model_ready_v2.csv')

print(f'📊 Extended: {df_ext.shape}')
print(f'🤖 Model-ready: {df_model.shape}')
print(f'📅 Range: {df_ext["year"].min()} – {df_ext["year"].max()}')
print(f'📍 Wells: {df_ext["well_id"].nunique()}, Districts: {df_ext["district"].nunique()}')

In [ ]:
# ============================================================
# 3. DEFINE FEATURES & TARGET
# ============================================================
TARGET = 'depth_mbgl'

FEATURES = [
    # Meteorological (time-varying)
    'rainfall_mm', 'temperature_avg', 'humidity', 'evapotranspiration',
    'soil_moisture_index',
    # Lag features (critical)
    'rainfall_lag_1m', 'rainfall_lag_2m', 'rainfall_lag_3m',
    'rainfall_rolling_3m', 'rainfall_rolling_6m',
    'rainfall_deficit', 'cumulative_deficit', 'temp_rainfall_ratio',
    # Groundwater lags
    'depth_lag_1q', 'depth_lag_2q', 'depth_change_rate',
    # Temporal
    'month', 'season_encoded',
    # Spatial
    'district_encoded', 'latitude', 'longitude',
    # Terrain
    'elevation_m', 'slope_degree', 'soil_type_encoded', 'ndvi'
]

print(f'🎯 Target: {TARGET}')
print(f'📋 Features ({len(FEATURES)}): {FEATURES}')

X = df_model[FEATURES].copy()
y = df_model[TARGET].copy()

print(f'\nX shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'y range: [{y.min():.2f}, {y.max():.2f}]')

In [ ]:
# ============================================================
# 4. TIME-BASED TRAIN/TEST SPLIT
# ============================================================
# Use year column from extended dataset
years = df_ext['year'].values

# Train: 2015-2023, Test: 2024-2025
train_mask = years <= 2023
test_mask = years >= 2024

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

print(f'📊 TIME-BASED SPLIT:')
print(f'  Train (2015–2023): {X_train.shape[0]:>8,} samples ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'  Test  (2024–2025): {X_test.shape[0]:>8,} samples ({X_test.shape[0]/len(X)*100:.1f}%)')
print(f'\n  Train y range: [{y_train.min():.2f}, {y_train.max():.2f}], mean={y_train.mean():.2f}')
print(f'  Test  y range: [{y_test.min():.2f}, {y_test.max():.2f}], mean={y_test.mean():.2f}')

In [ ]:
# ============================================================
# 5. FEATURE SCALING (for DL models)
# ============================================================
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).ravel()
y_test_scaled = scaler_y.transform(y_test.values.reshape(-1, 1)).ravel()

# Save scalers for production
joblib.dump(scaler_X, '../models/scaler_X.pkl')
joblib.dump(scaler_y, '../models/scaler_y.pkl')

print(f'✅ Scalers fitted and saved')
print(f'  X_train_scaled range: [{X_train_scaled.min():.2f}, {X_train_scaled.max():.2f}]')
print(f'  y_train_scaled range: [{y_train_scaled.min():.2f}, {y_train_scaled.max():.2f}]')

---
## 🌲 Model 1: XGBoost Regressor
Primary tabular model — expected R² > 0.95

In [ ]:
# ============================================================
# 6. XGBOOST MODEL
# ============================================================
print('\n' + '='*60)
print('  TRAINING XGBOOST REGRESSOR')
print('='*60)

xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbosity=1
)

# Train with early stopping
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50
)

# Predictions
y_pred_xgb_train = xgb_model.predict(X_train)
y_pred_xgb = xgb_model.predict(X_test)

print(f'\n✅ XGBoost Training Complete!')

In [ ]:
# ============================================================
# 7. XGBOOST EVALUATION
# ============================================================
def evaluate_model(y_true, y_pred, model_name='Model'):
    """Calculate and display comprehensive metrics."""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    
    # Risk classification accuracy
    def classify_risk(depth):
        if depth < 30: return 'Safe'
        elif depth < 100: return 'Warning'
        else: return 'Critical'
    
    true_risk = pd.Series(y_true).apply(classify_risk)
    pred_risk = pd.Series(y_pred).apply(classify_risk)
    risk_acc = (true_risk == pred_risk).mean() * 100
    
    metrics = {
        'model': model_name,
        'rmse': rmse,
        'mae': mae,
        'r2': r2,
        'mape': mape,
        'risk_accuracy': risk_acc
    }
    
    print(f'\n📊 {model_name} METRICS:')
    print(f'  RMSE:              {rmse:.4f} m')
    print(f'  MAE:               {mae:.4f} m')
    print(f'  R² Score:           {r2:.4f}')
    print(f'  MAPE:              {mape:.2f}%')
    print(f'  Risk Class. Acc:   {risk_acc:.2f}%')
    
    return metrics

xgb_metrics_train = evaluate_model(y_train.values, y_pred_xgb_train, 'XGBoost (Train)')
xgb_metrics = evaluate_model(y_test.values, y_pred_xgb, 'XGBoost (Test)')

In [ ]:
# ============================================================
# 8. XGBOOST — VISUALIZATIONS
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Actual vs Predicted
axes[0,0].scatter(y_test, y_pred_xgb, alpha=0.15, s=5, color=COLORS['deep'])
axes[0,0].plot([0, 350], [0, 350], 'r--', lw=2, label='Perfect prediction')
axes[0,0].set_xlabel('Actual Depth (m)')
axes[0,0].set_ylabel('Predicted Depth (m)')
axes[0,0].set_title(f'XGBoost: Actual vs Predicted (R²={xgb_metrics["r2"]:.4f})')
axes[0,0].legend()

# Residuals distribution
residuals = y_test.values - y_pred_xgb
axes[0,1].hist(residuals, bins=80, color=COLORS['mid'], edgecolor='white', alpha=0.8)
axes[0,1].axvline(0, color=COLORS['crisis'], ls='--', lw=2)
axes[0,1].set_xlabel('Residual (Actual - Predicted) m')
axes[0,1].set_ylabel('Frequency')
axes[0,1].set_title(f'Residuals Distribution (mean={residuals.mean():.3f}, std={residuals.std():.3f})')

# Feature Importance
importance = pd.Series(xgb_model.feature_importances_, index=FEATURES).sort_values(ascending=True)
importance.tail(15).plot(kind='barh', color=COLORS['deep'], ax=axes[1,0])
axes[1,0].set_title('XGBoost Feature Importance (Top 15)')
axes[1,0].set_xlabel('Importance Score')

# Prediction over time (sample)
test_dates = df_ext[test_mask]['date'].values
sample_well_mask = df_ext[test_mask]['well_id'] == df_ext[test_mask]['well_id'].iloc[0]
axes[1,1].plot(test_dates[sample_well_mask], y_test.values[sample_well_mask], label='Actual', color=COLORS['deep'], lw=2)
axes[1,1].plot(test_dates[sample_well_mask], y_pred_xgb[sample_well_mask], label='Predicted', color=COLORS['crisis'], ls='--', lw=2)
axes[1,1].set_title('XGBoost — Sample Well Predictions')
axes[1,1].set_ylabel('Depth (m)')
axes[1,1].invert_yaxis()
axes[1,1].legend()

plt.tight_layout()
plt.savefig('../data/plots/11_xgboost_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 🌲 Model 2: Random Forest Regressor

In [ ]:
# ============================================================
# 9. RANDOM FOREST MODEL
# ============================================================
print('\n' + '='*60)
print('  TRAINING RANDOM FOREST REGRESSOR')
print('='*60)

rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

rf_model.fit(X_train, y_train)

y_pred_rf_train = rf_model.predict(X_train)
y_pred_rf = rf_model.predict(X_test)

rf_metrics_train = evaluate_model(y_train.values, y_pred_rf_train, 'Random Forest (Train)')
rf_metrics = evaluate_model(y_test.values, y_pred_rf, 'Random Forest (Test)')

print(f'\n✅ Random Forest Training Complete!')

---
## 🧠 Model 3: LSTM Deep Learning Model
Sequence model using 12-month lookback windows.

In [ ]:
# ============================================================
# 10. PREPARE SEQUENCES FOR LSTM
# ============================================================
LOOKBACK = 12  # 12 months lookback

def create_sequences(X_data, y_data, well_ids, lookback=12):
    """Create sequences per well (respecting well boundaries)."""
    X_seq, y_seq = [], []
    unique_wells = well_ids.unique()
    
    for well in unique_wells:
        mask = well_ids == well
        X_well = X_data[mask]
        y_well = y_data[mask]
        
        if len(X_well) < lookback + 1:
            continue
        
        for i in range(lookback, len(X_well)):
            X_seq.append(X_well[i - lookback:i])
            y_seq.append(y_well[i])
    
    return np.array(X_seq), np.array(y_seq)

# Get well_ids for train/test
train_wells = df_ext[train_mask]['well_id'].values
test_wells = df_ext[test_mask]['well_id'].values

# Create sequences from scaled data
X_train_seq, y_train_seq = create_sequences(
    X_train_scaled, y_train_scaled,
    pd.Series(train_wells), LOOKBACK
)
X_test_seq, y_test_seq = create_sequences(
    X_test_scaled, y_test_scaled,
    pd.Series(test_wells), LOOKBACK
)

print(f'🧠 LSTM Sequence Shapes:')
print(f'  X_train_seq: {X_train_seq.shape}  (samples, timesteps, features)')
print(f'  y_train_seq: {y_train_seq.shape}')
print(f'  X_test_seq:  {X_test_seq.shape}')
print(f'  y_test_seq:  {y_test_seq.shape}')

In [ ]:
# ============================================================
# 11. BUILD & TRAIN LSTM MODEL
# ============================================================
print('\n' + '='*60)
print('  TRAINING LSTM MODEL')
print('='*60)

n_features = X_train_seq.shape[2]

lstm_model = Sequential([
    Input(shape=(LOOKBACK, n_features)),
    LSTM(128, return_sequences=True),
    Dropout(0.2),
    LSTM(64, return_sequences=False),
    Dropout(0.2),
    Dense(32, activation='relu'),
    BatchNormalization(),
    Dense(16, activation='relu'),
    Dense(1)
])

lstm_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

lstm_model.summary()

# Callbacks
callbacks = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1),
    ModelCheckpoint('../models/lstm_best.keras', monitor='val_loss', save_best_only=True)
]

# Train
history_lstm = lstm_model.fit(
    X_train_seq, y_train_seq,
    validation_split=0.15,
    epochs=100,
    batch_size=256,
    callbacks=callbacks,
    verbose=1
)

print(f'\n✅ LSTM Training Complete!')

In [ ]:
# ============================================================
# 12. LSTM TRAINING HISTORY
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history_lstm.history['loss'], label='Train Loss', color=COLORS['deep'], lw=2)
axes[0].plot(history_lstm.history['val_loss'], label='Val Loss', color=COLORS['crisis'], lw=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('LSTM Training Loss')
axes[0].legend()

# MAE
axes[1].plot(history_lstm.history['mae'], label='Train MAE', color=COLORS['deep'], lw=2)
axes[1].plot(history_lstm.history['val_mae'], label='Val MAE', color=COLORS['crisis'], lw=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].set_title('LSTM Training MAE')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/plots/12_lstm_training_history.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 13. LSTM EVALUATION
# ============================================================
# Predict on test sequences
y_pred_lstm_scaled = lstm_model.predict(X_test_seq, verbose=0).ravel()

# Inverse transform to original scale
y_pred_lstm = scaler_y.inverse_transform(y_pred_lstm_scaled.reshape(-1, 1)).ravel()
y_test_lstm = scaler_y.inverse_transform(y_test_seq.reshape(-1, 1)).ravel()

lstm_metrics = evaluate_model(y_test_lstm, y_pred_lstm, 'LSTM (Test)')

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test_lstm, y_pred_lstm, alpha=0.1, s=5, color=COLORS['deep'])
axes[0].plot([0, 350], [0, 350], 'r--', lw=2)
axes[0].set_xlabel('Actual Depth (m)')
axes[0].set_ylabel('Predicted Depth (m)')
axes[0].set_title(f'LSTM: Actual vs Predicted (R²={lstm_metrics["r2"]:.4f})')

residuals_lstm = y_test_lstm - y_pred_lstm
axes[1].hist(residuals_lstm, bins=80, color=COLORS['mid'], edgecolor='white')
axes[1].axvline(0, color=COLORS['crisis'], ls='--', lw=2)
axes[1].set_title(f'LSTM Residuals (std={residuals_lstm.std():.3f})')

plt.tight_layout()
plt.savefig('../data/plots/13_lstm_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 🧠 Model 4: GRU Model

In [ ]:
# ============================================================
# 14. GRU MODEL
# ============================================================
print('\n' + '='*60)
print('  TRAINING GRU MODEL')
print('='*60)

gru_model = Sequential([
    Input(shape=(LOOKBACK, n_features)),
    GRU(128, return_sequences=True),
    Dropout(0.2),
    GRU(64, return_sequences=False),
    Dropout(0.2),
    Dense(32, activation='relu'),
    BatchNormalization(),
    Dense(16, activation='relu'),
    Dense(1)
])

gru_model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

callbacks_gru = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1),
    ModelCheckpoint('../models/gru_best.keras', monitor='val_loss', save_best_only=True)
]

history_gru = gru_model.fit(
    X_train_seq, y_train_seq,
    validation_split=0.15,
    epochs=100,
    batch_size=256,
    callbacks=callbacks_gru,
    verbose=1
)

# Evaluate
y_pred_gru_scaled = gru_model.predict(X_test_seq, verbose=0).ravel()
y_pred_gru = scaler_y.inverse_transform(y_pred_gru_scaled.reshape(-1, 1)).ravel()

gru_metrics = evaluate_model(y_test_lstm, y_pred_gru, 'GRU (Test)')
print(f'\n✅ GRU Training Complete!')

---
## 🎯 Ensemble Model

In [ ]:
# ============================================================
# 15. ENSEMBLE MODEL
# ============================================================
print('\n' + '='*60)
print('  BUILDING ENSEMBLE MODEL')
print('='*60)

# For ensemble, we need aligned predictions on the same test set.
# XGBoost/RF predict on all test samples; LSTM/GRU predict on sequence-aligned subset.
# We'll create the ensemble on XGBoost + RF for the full test set.

# Weights based on individual R² performance
ENSEMBLE_WEIGHTS = {
    'xgboost': 0.45,
    'rf': 0.25,
    'lstm': 0.20,
    'gru': 0.10
}

# For the full test set (XGBoost + RF ensemble)
y_pred_ensemble_full = (
    ENSEMBLE_WEIGHTS['xgboost'] / (ENSEMBLE_WEIGHTS['xgboost'] + ENSEMBLE_WEIGHTS['rf']) * y_pred_xgb +
    ENSEMBLE_WEIGHTS['rf'] / (ENSEMBLE_WEIGHTS['xgboost'] + ENSEMBLE_WEIGHTS['rf']) * y_pred_rf
)

ensemble_full_metrics = evaluate_model(y_test.values, y_pred_ensemble_full, 'Ensemble-Full (XGB+RF)')

print(f'\nEnsemble weights: {ENSEMBLE_WEIGHTS}')

---
## 📊 Model Comparison

In [ ]:
# ============================================================
# 16. MODEL COMPARISON
# ============================================================
all_metrics = [
    xgb_metrics, rf_metrics, lstm_metrics, gru_metrics, ensemble_full_metrics
]
comparison_df = pd.DataFrame(all_metrics)
comparison_df = comparison_df.set_index('model')

print('\n' + '='*80)
print('  🏆 MODEL COMPARISON RESULTS')
print('='*80)
display(comparison_df.style.highlight_max(axis=0, subset=['r2', 'risk_accuracy'], color='#c6efce')
        .highlight_min(axis=0, subset=['rmse', 'mae', 'mape'], color='#c6efce')
        .format({'rmse': '{:.4f}', 'mae': '{:.4f}', 'r2': '{:.4f}',
                 'mape': '{:.2f}%', 'risk_accuracy': '{:.2f}%'}))

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

models = comparison_df.index.tolist()
colors = [COLORS['deep'], COLORS['safe'], COLORS['mid'], COLORS['warn'], COLORS['crisis']]

# R² comparison
axes[0].barh(models, comparison_df['r2'], color=colors)
axes[0].set_xlabel('R² Score')
axes[0].set_title('🏆 R² Score Comparison')
axes[0].set_xlim(0.8, 1.0)
for i, v in enumerate(comparison_df['r2']):
    axes[0].text(v + 0.002, i, f'{v:.4f}', va='center', fontweight='bold')

# RMSE comparison
axes[1].barh(models, comparison_df['rmse'], color=colors)
axes[1].set_xlabel('RMSE (m)')
axes[1].set_title('📉 RMSE Comparison (lower is better)')
for i, v in enumerate(comparison_df['rmse']):
    axes[1].text(v + 0.2, i, f'{v:.2f}', va='center', fontweight='bold')

# Risk accuracy
axes[2].barh(models, comparison_df['risk_accuracy'], color=colors)
axes[2].set_xlabel('Risk Classification Accuracy (%)')
axes[2].set_title('⚠️ Risk Classification Accuracy')
for i, v in enumerate(comparison_df['risk_accuracy']):
    axes[2].text(v + 0.3, i, f'{v:.1f}%', va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../data/plots/14_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 17. SHAP FEATURE IMPORTANCE (XGBoost)
# ============================================================
try:
    import shap
    
    print('Computing SHAP values (this may take a minute)...')
    explainer = shap.TreeExplainer(xgb_model)
    
    # Use a sample for speed
    X_sample = X_test.sample(min(2000, len(X_test)), random_state=42)
    shap_values = explainer.shap_values(X_sample)
    
    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    
    plt.sca(axes[0])
    shap.summary_plot(shap_values, X_sample, plot_type='bar', show=False,
                      max_display=15)
    axes[0].set_title('SHAP Feature Importance (Bar)')
    
    plt.sca(axes[1])
    shap.summary_plot(shap_values, X_sample, show=False, max_display=15)
    axes[1].set_title('SHAP Summary Plot')
    
    plt.tight_layout()
    plt.savefig('../data/plots/15_shap_importance.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ SHAP analysis complete!')
    
except ImportError:
    print('⚠️ SHAP not installed. Install with: pip install shap')
    print('Skipping SHAP analysis.')

---
## 📦 Export Models for Production

In [ ]:
# ============================================================
# 18. SAVE ALL MODELS
# ============================================================
print('\n' + '='*60)
print('  SAVING MODELS FOR PRODUCTION')
print('='*60)

# 1. XGBoost
joblib.dump(xgb_model, '../models/xgboost_model.pkl')
print('✅ XGBoost saved → models/xgboost_model.pkl')

# 2. Random Forest
joblib.dump(rf_model, '../models/rf_model.pkl')
print('✅ Random Forest saved → models/rf_model.pkl')

# 3. LSTM
lstm_model.save('../models/lstm_model.keras')
print('✅ LSTM saved → models/lstm_model.keras')

# 4. GRU
gru_model.save('../models/gru_model.keras')
print('✅ GRU saved → models/gru_model.keras')

# 5. Scalers (already saved)
print('✅ Scalers already saved → models/scaler_X.pkl, scaler_y.pkl')

# 6. Feature list & config
model_config = {
    'features': FEATURES,
    'target': TARGET,
    'lookback': LOOKBACK,
    'ensemble_weights': ENSEMBLE_WEIGHTS,
    'risk_thresholds': {
        'safe': [0, 30],
        'warning': [30, 100],
        'critical': [100, 200],
        'extreme': [200, 500]
    },
    'metrics': {
        'xgboost': xgb_metrics,
        'random_forest': rf_metrics,
        'lstm': lstm_metrics,
        'gru': gru_metrics,
        'ensemble': ensemble_full_metrics
    },
    'training_date': datetime.now().isoformat(),
    'dataset_shape': list(df_model.shape),
    'train_size': int(X_train.shape[0]),
    'test_size': int(X_test.shape[0])
}

with open('../models/model_config.json', 'w') as f:
    json.dump(model_config, f, indent=2, default=str)
print('✅ Config saved → models/model_config.json')

# 7. Save feature names for the API
with open('../models/features.json', 'w') as f:
    json.dump({'features': FEATURES, 'target': TARGET}, f, indent=2)
print('✅ Feature list saved → models/features.json')

print(f'\n🎉 All models and artifacts exported successfully!')

# List saved files
print(f'\n📁 Models directory contents:')
for f in sorted(os.listdir('../models')):
    size = os.path.getsize(f'../models/{f}') / 1024
    unit = 'KB' if size < 1024 else 'MB'
    size = size if size < 1024 else size / 1024
    print(f'  {f:40s} {size:>8.1f} {unit}')

In [ ]:
# ============================================================
# 19. FINAL TRAINING REPORT
# ============================================================
best_model = comparison_df['r2'].idxmax()
best_r2 = comparison_df.loc[best_model, 'r2']

print('\n' + '='*70)
print('  🏆 FINAL TRAINING REPORT')
print('='*70)
print(f'''
🏆 BEST MODEL: {best_model} (R² = {best_r2:.4f})

📊 MODELS TRAINED:
  1. XGBoost       → R²={xgb_metrics["r2"]:.4f}  RMSE={xgb_metrics["rmse"]:.2f}m  Risk Acc={xgb_metrics["risk_accuracy"]:.1f}%
  2. Random Forest  → R²={rf_metrics["r2"]:.4f}  RMSE={rf_metrics["rmse"]:.2f}m  Risk Acc={rf_metrics["risk_accuracy"]:.1f}%
  3. LSTM           → R²={lstm_metrics["r2"]:.4f}  RMSE={lstm_metrics["rmse"]:.2f}m  Risk Acc={lstm_metrics["risk_accuracy"]:.1f}%
  4. GRU            → R²={gru_metrics["r2"]:.4f}  RMSE={gru_metrics["rmse"]:.2f}m  Risk Acc={gru_metrics["risk_accuracy"]:.1f}%
  5. Ensemble       → R²={ensemble_full_metrics["r2"]:.4f}  RMSE={ensemble_full_metrics["rmse"]:.2f}m

📦 EXPORTED ARTIFACTS:
  • models/xgboost_model.pkl     - Primary prediction model
  • models/rf_model.pkl          - Random Forest backup
  • models/lstm_model.keras      - Deep Learning sequence model
  • models/gru_model.keras       - GRU sequence model
  • models/scaler_X.pkl          - Feature scaler
  • models/scaler_y.pkl          - Target scaler
  • models/model_config.json     - Full configuration
  • models/features.json         - Feature list

✅ READY FOR FASTAPI BACKEND DEPLOYMENT!
''')